# Idiom Fix Experiments: HIMYM `s04e12_benefits_l1055_1259`

This notebook tests idiom-specific fixes on top of the existing run output in [`latest_predictions.csv`](../manual_tests/datasets/himym/s04e12_benefits_l1055_1259/latest_predictions.csv).

Important scope note:
- this is a **post-filtering experiment** over the emitted idiom predictions
- it does **not** rerun the full MWE engine
- that is enough for the current question, because the main problem is idiom precision, not idiom recall

Each experiment cell implements one mitigation and shows its effect on idiom-only metrics.


In [1]:
from pathlib import Path
import csv
import re
from collections import Counter, defaultdict
from itertools import combinations

DATASET_DIR = Path("../manual_tests/datasets/himym/s04e12_benefits_l1055_1259")
PRED_PATH = DATASET_DIR / "latest_predictions.csv"
GOLD_PATH = DATASET_DIR / "gold_v.1.0.csv"

PLACEHOLDER_TOKENS = {
    "someone", "something", "some", "someplace", "place", "animal", "oneself"
}

FUNCTION_TOKENS = {
    "a", "an", "the", "at", "by", "for", "from", "in", "into", "of", "on",
    "out", "to", "up", "with", "it", "there", "here", "what", "more"
}

# Small exploratory blocklist based on the FP analysis notebook.
# These are generic/default-dangerous idiom entries for this slice.
GENERIC_BLOCKLIST = {
    "at work",
    "do what",
    "for good",
    "go there",
    "in addition to something",
    "in there",
    "more and more",
    "no go",
    "time for someone or something",
    "what happened",
}

def load_csv(path: Path):
    with path.open(newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

def normalize_expr(text: str) -> str:
    return re.sub(r"[^a-z0-9 ]+", "", text.lower()).strip()

def tokenize(text: str):
    return re.findall(r"[a-z0-9']+", text.lower())

def normalized_surface(text: str) -> str:
    return " ".join(tokenize(text))

def pct(part, whole):
    return 0.0 if whole == 0 else round(part * 100 / whole, 1)

def precision(tp, pred):
    return 0.0 if pred == 0 else tp / pred

def recall(tp, gold):
    return 0.0 if gold == 0 else tp / gold

def f1(p, r):
    return 0.0 if p + r == 0 else 2 * p * r / (p + r)

def show_metrics(label, metrics):
    print(label)
    print(f"  predicted:  {metrics['pred']}")
    print(f"  tp / fp / fn: {metrics['tp']} / {metrics['fp']} / {metrics['fn']}")
    print(f"  precision: {metrics['precision']:.3f}")
    print(f"  recall:    {metrics['recall']:.3f}")
    print(f"  f1:        {metrics['f1']:.3f}")

rows = load_csv(PRED_PATH)
gold_rows = load_csv(GOLD_PATH)

idiom_gold = {
    (int(r["line_index"]), normalize_expr(r["expression"]))
    for r in gold_rows
    if r["expression_type"] == "idiom"
}

idiom_predictions = []
for row in rows:
    if row.get("predicted") != "yes":
        continue
    if row.get("prediction_type") != "idiom":
        continue
    idiom_predictions.append({
        "line_index": int(row["line_index"]),
        "speaker": row["speaker"],
        "line": row["line"],
        "expr": row["prediction_expression"],
        "expr_norm": normalize_expr(row["prediction_expression"]),
    })

def metrics_for(predictions):
    pred_set = {(r['line_index'], r['expr_norm']) for r in predictions}
    tp = len(pred_set & idiom_gold)
    fp = len(pred_set - idiom_gold)
    fn = len(idiom_gold - pred_set)
    p = precision(tp, len(pred_set))
    r = recall(tp, len(idiom_gold))
    return {
        "pred": len(pred_set),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": p,
        "recall": r,
        "f1": f1(p, r),
    }

def clip(text, width=110):
    text = " ".join(text.split())
    return text if len(text) <= width else text[: width - 3] + "..."

baseline_metrics = metrics_for(idiom_predictions)
show_metrics("Baseline idiom metrics", baseline_metrics)
print()
print(f"Gold idioms:      {len(idiom_gold)}")
print(f"Predicted idioms: {baseline_metrics['pred']}")
print(f"Unique FP idioms: {baseline_metrics['fp']}")
print(f"Unique TP idioms: {baseline_metrics['tp']}")


Baseline idiom metrics
  predicted:  62
  tp / fp / fn: 4 / 58 / 10
  precision: 0.065
  recall:    0.286
  f1:        0.105

Gold idioms:      14
Predicted idioms: 62
Unique FP idioms: 58
Unique TP idioms: 4


In [2]:
print("Most frequent predicted idiom expressions:")
for expr, count in Counter(r['expr_norm'] for r in idiom_predictions).most_common(20):
    print(f"  {count:>2}  {expr}")

print("\nLines with the most idiom predictions:")
by_line = defaultdict(list)
for row in idiom_predictions:
    by_line[row['line_index']].append(row)

for line_index, items in sorted(by_line.items(), key=lambda kv: (-len(kv[1]), kv[0]))[:10]:
    print(f"  line {line_index}: {len(items)} predictions | {clip(items[0]['line'], 120)}")


Most frequent predicted idiom expressions:
   2  get out of here
   2  get something out
   2  take someone in
   2  take someone or an animal in
   2  take someone something or an animal inside
   2  take something in
   1  take someone or something out
   1  take someone out
   1  take something out
   1  couple something together
   1  go there
   1  no go
   1  what happened
   1  live together
   1  boil down to something
   1  boil something down
   1  boil down
   1  do what
   1  in there
   1  read something in something

Lines with the most idiom predictions:
  line 1209: 9 predictions | Because of your bickering roommates are always a source of conflict between you two, I wanted to help. In fact, I wen...
  line 1255: 8 predictions | Right. It's not like you, you know? In addition, we are friends. I want to complicate matters by committing. Hanging ...
  line 1068: 5 predictions | No, I went there yesterday... Stop it! Stop it! My God, what happens? When we were a couple, we

## Solution 1: remove placeholder-heavy idiom patterns

Motivation:
- many false positives come from entries like `take someone in`, `hang out with someone or something`, `room with someone`
- these are slot-heavy patterns that behave more like templates than fixed idioms
- the lowest-risk first pass is to remove any predicted idiom whose canonical expression contains placeholder tokens such as `someone`, `something`, `some`, `animal`


In [3]:
def has_placeholder(expr: str) -> bool:
    tokens = tokenize(expr)
    return any(token in PLACEHOLDER_TOKENS for token in tokens)

placeholder_filtered = [r for r in idiom_predictions if not has_placeholder(r['expr'])]
placeholder_metrics = metrics_for(placeholder_filtered)
show_metrics("After placeholder filter", placeholder_metrics)

removed_placeholder = [r for r in idiom_predictions if has_placeholder(r['expr'])]
print("\nRemoved examples:")
for row in removed_placeholder[:12]:
    print(f"  line {row['line_index']} | {row['expr']} | {clip(row['line'])}")


After placeholder filter
  predicted:  24
  tp / fp / fn: 4 / 20 / 10
  precision: 0.167
  recall:    0.286
  f1:        0.211

Removed examples:
  line 1061 | take someone or something out | So, take out the trash.
  line 1061 | take someone out | So, take out the trash.
  line 1061 | take something out | So, take out the trash.
  line 1068 | couple something together | No, I went there yesterday... Stop it! Stop it! My God, what happens? When we were a couple, we lived toget...
  line 1072 | boil down to something | I explained. I said, Madeline, every international conflict essentially boils down to sexual tension.
  line 1072 | boil something down | I explained. I said, Madeline, every international conflict essentially boils down to sexual tension.
  line 1091 | read something in something | I was working and I had to take a leap here...reading this magazine. In... the room there.
  line 1093 | room with someone | If this is a problem. You've done all the way here to read a magazi

## Solution 2: block a small set of broad / generic idiom entries

Motivation:
- some idiom entries are simply too generic to fire by default
- examples on this slice: `at work`, `go there`, `more and more`, `no go`, `what happened`
- this cell applies a small exploratory blocklist derived from the earlier FP analysis

This is intentionally conservative. It is not trying to solve the whole problem alone.


In [4]:
def is_generic(expr: str) -> bool:
    return normalize_expr(expr) in GENERIC_BLOCKLIST

generic_filtered = [r for r in idiom_predictions if not is_generic(r['expr'])]
generic_metrics = metrics_for(generic_filtered)
show_metrics("After generic-entry blocklist", generic_metrics)

print("\nBlocked entries:")
for expr in sorted(GENERIC_BLOCKLIST):
    count = sum(1 for r in idiom_predictions if r['expr_norm'] == expr)
    if count:
        print(f"  {expr:28} {count}")


After generic-entry blocklist
  predicted:  52
  tp / fp / fn: 4 / 48 / 10
  precision: 0.077
  recall:    0.286
  f1:        0.121

Blocked entries:
  at work                      1
  do what                      1
  for good                     1
  go there                     1
  in addition to something     1
  in there                     1
  more and more                1
  no go                        1
  time for someone or something 1
  what happened                1


## Solution 3: per-line ranking as an approximation of span deduplication

Motivation:
- one real local phrase often triggers several neighboring idiom entries
- the predictions CSV does not store exact token spans, so we cannot do true span clustering here
- instead, we approximate it by scoring each prediction in a line and keeping only the top `k`

Scoring heuristic:
- reward content words
- reward exact surface appearance inside the line
- penalize placeholder-heavy expressions
- penalize high function-word ratio


In [5]:
def exact_surface_match(expr: str, line: str) -> bool:
    return normalized_surface(expr) in normalized_surface(line)

def score_prediction(row) -> int:
    tokens = tokenize(row['expr'])
    content_count = sum(token not in PLACEHOLDER_TOKENS and token not in FUNCTION_TOKENS for token in tokens)
    function_ratio = 0.0 if not tokens else sum(token in FUNCTION_TOKENS for token in tokens) / len(tokens)

    score = content_count * 3 + len(tokens)
    if exact_surface_match(row['expr'], row['line']):
        score += 4
    if has_placeholder(row['expr']):
        score -= 6
    if function_ratio >= 0.6:
        score -= 3
    return score

def keep_top_k_per_line(predictions, k: int):
    grouped = defaultdict(list)
    for row in predictions:
        grouped[row['line_index']].append(row)

    kept = []
    for line_index, items in grouped.items():
        ranked = sorted(items, key=score_prediction, reverse=True)
        kept.extend(ranked[:k])
    return kept

for k in [1, 2, 3]:
    ranked_predictions = keep_top_k_per_line(idiom_predictions, k)
    ranked_metrics = metrics_for(ranked_predictions)
    show_metrics(f"After per-line top-{k} ranking", ranked_metrics)
    print()

print("Example ranking on a noisy line:")
sample_line = 1255
for row in sorted(by_line[sample_line], key=score_prediction, reverse=True):
    print(f"  score={score_prediction(row):>2} | {row['expr']:35} | {clip(row['line'], 95)}")


After per-line top-1 ranking
  predicted:  20
  tp / fp / fn: 2 / 18 / 12
  precision: 0.100
  recall:    0.143
  f1:        0.118

After per-line top-2 ranking
  predicted:  35
  tp / fp / fn: 4 / 31 / 10
  precision: 0.114
  recall:    0.286
  f1:        0.163

After per-line top-3 ranking
  predicted:  44
  tp / fp / fn: 4 / 40 / 10
  precision: 0.091
  recall:    0.286
  f1:        0.138

Example ranking on a noisy line:
  score=12 | like you know                       | Right. It's not like you, you know? In addition, we are friends. I want to complicate matter...
  score= 6 | hang out with someone or something  | Right. It's not like you, you know? In addition, we are friends. I want to complicate matter...
  score= 6 | hang someone or something with something | Right. It's not like you, you know? In addition, we are friends. I want to complicate matter...
  score= 2 | hang something out of something     | Right. It's not like you, you know? In addition, we are friends. I want to

## Combine the solutions

Now we evaluate combinations of:
- placeholder filter
- generic-entry blocklist
- line-level ranking (`top-1`, `top-2`, `top-3`)

This lets us find the best precision / recall / F1 tradeoff for this slice.


In [6]:
def apply_placeholder_filter(predictions):
    return [r for r in predictions if not has_placeholder(r['expr'])]

def apply_generic_filter(predictions):
    return [r for r in predictions if not is_generic(r['expr'])]

strategy_results = []

base_variants = {
    "baseline": idiom_predictions,
    "placeholder_only": apply_placeholder_filter(idiom_predictions),
    "generic_only": apply_generic_filter(idiom_predictions),
    "placeholder_plus_generic": apply_generic_filter(apply_placeholder_filter(idiom_predictions)),
}

for name, preds in base_variants.items():
    strategy_results.append((name, metrics_for(preds), preds))
    for k in [1, 2, 3]:
        ranked = keep_top_k_per_line(preds, k)
        strategy_results.append((f"{name}_top{k}", metrics_for(ranked), ranked))

strategy_results.sort(key=lambda item: (item[1]['f1'], item[1]['precision'], -item[1]['fp']), reverse=True)

print("Strategy ranking:")
for name, metrics, _ in strategy_results:
    print(
        f"  {name:30} pred={metrics['pred']:>2}  tp={metrics['tp']:>2}  fp={metrics['fp']:>2}  fn={metrics['fn']:>2}  "
        f"P={metrics['precision']:.3f}  R={metrics['recall']:.3f}  F1={metrics['f1']:.3f}"
    )

best_name, best_metrics, best_predictions = strategy_results[0]
print("\nBest strategy on this slice:")
show_metrics(best_name, best_metrics)


Strategy ranking:
  placeholder_plus_generic_top1  pred=13  tp= 4  fp= 9  fn=10  P=0.308  R=0.286  F1=0.296
  placeholder_plus_generic_top2  pred=15  tp= 4  fp=11  fn=10  P=0.267  R=0.286  F1=0.276
  placeholder_only_top1          pred=16  tp= 4  fp=12  fn=10  P=0.250  R=0.286  F1=0.267
  placeholder_plus_generic       pred=16  tp= 4  fp=12  fn=10  P=0.250  R=0.286  F1=0.267
  placeholder_plus_generic_top3  pred=16  tp= 4  fp=12  fn=10  P=0.250  R=0.286  F1=0.267
  placeholder_only_top2          pred=20  tp= 4  fp=16  fn=10  P=0.200  R=0.286  F1=0.235
  placeholder_only_top3          pred=23  tp= 4  fp=19  fn=10  P=0.174  R=0.286  F1=0.216
  placeholder_only               pred=24  tp= 4  fp=20  fn=10  P=0.167  R=0.286  F1=0.211
  generic_only_top2              pred=31  tp= 4  fp=27  fn=10  P=0.129  R=0.286  F1=0.178
  baseline_top2                  pred=35  tp= 4  fp=31  fn=10  P=0.114  R=0.286  F1=0.163
  generic_only_top3              pred=39  tp= 4  fp=35  fn=10  P=0.103  R=0.286  F

In [7]:
best_pred_set = {(r['line_index'], r['expr_norm']) for r in best_predictions}
best_tp = sorted(best_pred_set & idiom_gold)
best_fp = sorted(best_pred_set - idiom_gold)
best_fn = sorted(idiom_gold - best_pred_set)

line_lookup = {}
for row in idiom_predictions:
    line_lookup.setdefault(row['line_index'], row['line'])
for row in gold_rows:
    line_lookup.setdefault(int(row['line_index']), row['line'])

print("Retained true positives:")
for line_index, expr in best_tp:
    print(f"  line {line_index} | {expr:20} | {clip(line_lookup[line_index])}")

print("\nRemaining false positives:")
for line_index, expr in best_fp:
    print(f"  line {line_index} | {expr:28} | {clip(line_lookup[line_index])}")

print("\nStill-missed gold idioms:")
for line_index, expr in best_fn:
    print(f"  line {line_index} | {expr:20} | {clip(line_lookup[line_index])}")


Retained true positives:
  line 1093 | all the way          | If this is a problem. You've done all the way here to read a magazine? I am willing to bet that there is a ...
  line 1209 | in fact              | Because of your bickering roommates are always a source of conflict between you two, I wanted to help. In f...
  line 1211 | in passing           | I took it in passing. It's nothing.
  line 1245 | by the way           | I go step-louse. If you are looking for Ted, he was released. And... our little arrangement is... completed...

Remaining false positives:
  line 1068 | live together                | No, I went there yesterday... Stop it! Stop it! My God, what happens? When we were a couple, we lived toget...
  line 1072 | boil down                    | I explained. I said, Madeline, every international conflict essentially boils down to sexual tension.
  line 1164 | get it out                   | No, it's wrong. You must learn to get it out. As we did in my kindergarten class. 

## Takeaways

On this slice, the best experimental strategy is the one that combines:
- placeholder filtering
- a small generic-entry blocklist
- aggressive line-level ranking (`top-1`)

Why it wins here:
- placeholder filtering removes the biggest false-positive bucket with little risk
- the generic-entry blocklist removes broad low-value entries
- `top-1` ranking acts as a practical dedupe step and prevents one line from exploding into many idiom predictions

What this does **not** solve:
- the 10 remaining false negatives
- true syntax-aware idiom validation
- generalization beyond this slice

So the notebook should be read as a decision aid:
- **good for choosing the first production filters**
- **not yet a replacement for a cleaner idiom matcher**
